# Physical Pain
notebook containing experiments, preps & wrangling of physical pain data ([IHME](https://www.healthdata.org/sites/default/files/2025-10/GBD_2023_Booklet_Final_2025.10.17.pdf))


In [ ]:
# Imports
from typing import Dict, List, Optional, Set, Tuple
import gzip
import os
import pandas as pd

In [ ]:
# Path constants
BASE_PATH = os.path.join('..', 'data')
# filtered and partially aggregated physical dataset (otherwise it would be too detailed/big)
DATASET_PATH_RAW = os.path.join(BASE_PATH, 'raw', 'physical', 'prevalence_by_pixel.csv.gz')
DATASET_PATH_FILTERED = os.path.join(BASE_PATH, 'raw', 'physical', 'percent_filtered.csv.gz')
DATASET_PATH_AGGR_AVG = os.path.join(BASE_PATH, 'raw', 'physical', 'aggr_01x01_avg.csv.gz')
DATASET_PATH_AGGR_SUM = os.path.join(BASE_PATH, 'raw', 'physical', 'aggr_01x01_sum.csv.gz')
DATASET_PATH = DATASET_PATH_FILTERED
OUTPUT_PATH = os.path.join(BASE_PATH, "actual", "physical", "physical.csv")
DB_LOAD_PATH = os.path.join(BASE_PATH, "dummy", "data_types", "phys.csv")

In [ ]:
CHUNK_SIZE = 100_000

In [ ]:
# Quick sanity check
nrows = 10
dataset = pd.read_csv(
    DATASET_PATH,
    nrows=nrows,
    compression="gzip",   # <-- fix for UnicodeDecodeError
    #index_col="id"
)
dataset.head(n=nrows)

In [ ]:
# infer data types
sample = pd.read_csv(DATASET_PATH, nrows=10_000, compression="gzip")
print(sample.dtypes)

In [ ]:
# check avg aggregation
df_avg = pd.read_csv(DATASET_PATH_AGGR_AVG, nrows=10, compression="gzip")
df_avg.head(10)

In [ ]:

df_avg = pd.read_csv(DATASET_PATH_AGGR_SUM, nrows=10, skiprows=0, compression="gzip")
df_avg.head(10)

## Category Distributions

### Raw (runs for 22 min)
- 'Headache disorders':     361 790 800
- 'Rheumatoid arthritis':   361 790 800
- 'Osteoarthritis':         361 790 800
- 'Low back pain':          361 790 800
- 'Neck pain':              361 790 800
- 'Migraine':               361 046 000
Note: Migraine count is incomplete since I forgot to print the final count (so the last up to 99 chunks are not included)

### Filtered (runs for 7 min)
- 'Headache disorders':     180 895 400
- 'Rheumatoid arthritis':   180 895 400
- 'Osteoarthritis':         180 895 400
- 'Low back pain':          180 895 400
- 'Neck pain':              180 895 400
- 'Migraine':               175 523 000
Note: Migraine count is incomplete since I forgot to print the final count (so the last up to 99 chunks are not included)

### Aggregated
- 'Headache disorders':       1 339 885
- 'Rheumatoid arthritis':     1 339 885
- 'Osteoarthritis':           1 339 885
- 'Low back pain':            1 339 885
- 'Neck pain':                1 339 885
- 'Migraine':                 1 339 885

In [ ]:
# check listed categories
from collections import Counter

category_dataset_path = DATASET_PATH_AGGR_AVG
category_name = {
    DATASET_PATH_RAW: "cause_name",
    DATASET_PATH_FILTERED: "cause_name",
    DATASET_PATH_AGGR_AVG: "category",
}[category_dataset_path]

cause_counts = Counter()
seen_causes = set()
i = 0
for chunk in pd.read_csv(category_dataset_path, chunksize=CHUNK_SIZE, compression="gzip"):
    chunk_counts = chunk[category_name].dropna().astype(str).value_counts()

    for cause, count in chunk_counts.items():
        cause_counts[cause] += int(count)

        if cause not in seen_causes:
            seen_causes.add(cause)
            print(dict(cause_counts))

    i += 1
    if i % 100 == 0:
        print(dict(cause_counts))

print("Final counts:")
print(dict(cause_counts))

## Create Physical Pain Dataset

In [ ]:
# normalization / pain value computation
def _compute_pain(category_values: pd.Series, use_log: bool = False) -> pd.Series:
    max_pain = category_values.max()
    min_pain = category_values.min()
    assert min_pain >= 0

    if use_log:
      log_max = np.log(max_pain)
      log_min = np.log(min_pain)
      pain_range = log_max - log_min
      pain_offset = log_min
      #assert pain_range > 0, f"Invalid pain_range! log_max = {log_max}, log_min = {log_min}, pain_range = {pain_range}"
    else:
       pain_range = max_pain - min_pain
       pain_offset = min_pain

    return (
        (np.log(category_values) if use_log else category_values - pain_offset) / pain_range
    ).clip(0, 1).round(5)

def normalize_dataset(df: pd.DataFrame, categories: Optional[List[str]] = None, use_log: bool = False) -> pd.DataFrame:
    if categories is None:
        categories = sorted(df["category"].dropna().astype(str).unique())

    required_columns = {"value", "category", "lat", "lng"}
    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        raise KeyError(f"Missing required columns: {sorted(missing_columns)}")

    work_df = df.loc[:, ["value", "category", "lat", "lng"]].copy()
    work_df = work_df.dropna(subset=["value", "category", "lat", "lng"])
    work_df["category"] = work_df["category"].astype(str)

    normalized_frames = []

    for category in categories:
        category_df = work_df.loc[work_df["category"] == str(category)].copy()
        if category_df.empty:
            continue

        if category_df["value"].max() == category_df["value"].min():
            normalized_values = pd.Series(0.0, index=category_df.index)
        else:
            normalized_values = _compute_pain(category_df["value"], use_log)

        normalized_frames.append(pd.DataFrame({
            "aggrId": pd.Series([None] * len(category_df), index=category_df.index, dtype="Float64"),
            "value": normalized_values,
            "category": category_df["category"],
            "lat": category_df["lat"],
            "lng": category_df["lng"],
        }))

    if not normalized_frames:
        empty_df = pd.DataFrame({
            "aggrId": pd.Series(dtype="Float64"),
            "value": pd.Series(dtype="float64"),
            "category": pd.Series(dtype="object"),
            "lat": pd.Series(dtype="float64"),
            "lng": pd.Series(dtype="float64"),
        })
        empty_df.index = pd.RangeIndex(start=1, stop=1)
        return empty_df

    result = pd.concat(normalized_frames, ignore_index=True)
    result.index = result.index + 1
    return result

In [ ]:
# load dataset
my_dataset_path = os.path.join(BASE_PATH, "raw", "physical", "aggr_02x04_sum.csv.gz")
dataset = pd.read_csv(my_dataset_path, compression="gzip")
dataset.head()

In [ ]:
# raw value distribution
category = "Headache disorders"
df_cat = dataset[dataset["category"] == category]
print("max = ", df_cat["value"].max())
print("median = ", df_cat["value"].median())
print("avg = ", df_cat["value"].sum() / df_cat.size)
print("min = ", df_cat["value"].min())

In [ ]:
# extract categories
# categories = sorted(dataset["category"].dropna().astype(str).unique())
categories = ['Headache disorders', 'Low back pain', 'Migraine', 'Neck pain', 'Osteoarthritis', 'Rheumatoid arthritis']
print(categories)

In [ ]:
df_pain = normalize_dataset(dataset, categories=categories, use_log=False)
#print(sorted(df_pain["category"].dropna().astype(str).unique()))

In [ ]:
df_pain.to_csv(DB_LOAD_PATH, index=True, index_label="id")

# Plot data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Config
categories = ['Headache disorders', 'Low back pain', 'Migraine', 'Neck pain', 'Osteoarthritis', 'Rheumatoid arthritis']
# value bins: 0.0-0.1, 0.1-0.2, ..., 0.9-1.0
bins = np.linspace(0, 1, 11)
labels = [f"{bins[i]:.1f}-{bins[i + 1]:.1f}" for i in range(len(bins) - 1)]

In [ ]:
# plot category bins (FAULTY)
df_plot = df_pain.copy()
#df_plot = pd.read_csv(os.path.join(BASE_PATH, "raw", "physical", "phys_aio.csv"))

df_plot["value"] = pd.to_numeric(df_plot["value"], errors="coerce")
df_plot = df_plot.dropna(subset=["value", "category"])

n = len(categories)
fig, axes = plt.subplots(n, 1, figsize=(10, 2.8 * n), sharex=True)

for ax, cat in zip(axes, categories):
    s = df_plot.loc[df_plot["category"] == cat, "value"]

    # count rows in each bin
    counts = pd.cut(
        s,
        bins=bins,
        include_lowest=True,
        right=True
    ).value_counts(sort=False)

    # make sure all bins are present
    counts = counts.reindex(pd.IntervalIndex.from_breaks(bins), fill_value=0).astype(int)

    ax.bar(range(len(counts)), counts.values, color="#4C78A8")
    ax.set_title(cat, fontsize=10)
    ax.set_ylabel("rows")
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45)
    ax.set_ylim(0, max(1, counts.max() + 1))

fig.supxlabel("value range")
fig.supylabel("row count")
plt.tight_layout()
plt.show()

In [ ]:
cat_index = 0
#df_dist = pd.read_csv(os.path.join(BASE_PATH, "raw", "physical", "phys_aio.csv.gz"))
df_dist = pd.read_csv(os.path.join(BASE_PATH, "dummy", "data_types", "phys.csv"))
df_dist = df_dist[df_dist["aggrId"].isna()]
#df_dist = df_pain
df_cat = df_dist[df_dist["category"] == categories[cat_index]]
plt.plot(sorted(df_cat["value"]))

print("max = ", df_cat["value"].max())
print("median = ", df_cat["value"].median())
print("avg = ", df_cat["value"].sum() / df_cat.size)
print("min = ", df_cat["value"].min())

# Utility

In [ ]:
# rename column names to match our structure (expected time: 35 min for 12 GB file)
from shutil import copyfileobj
def rename_columns(src: str, dst: str):
    with gzip.open(src, "rt", encoding="utf-8", newline="") as inf, gzip.open(dst, "wt", encoding="utf-8", newline="") as outf:
        header = inf.readline().rstrip("\n").split(",")
        print(f"header = {', '.join(header)}")
        header = [
            "lng" if col == "lon" else
            "category" if col == "cause_name" else
            "value" if col.startswith("pixel_abs_prevalence") else  # probably some whitespace issue cause "==" doesn't work here
            col
            for col in header
        ]
        outf.write(",".join(header) + "\n")
        copyfileobj(inf, outf)


In [ ]:

src = os.path.join(BASE_PATH, "raw", "physical", "percent_filtered.csv.gz")
dst = os.path.join(BASE_PATH, "raw", "physical", "percent_filtered_renamed.csv.gz")
#rename_columns(src, dst)

## Compare values between original and renamed columns

In [ ]:
ROW_NUM = 10_000
df_lon = pd.read_csv(
  os.path.join(BASE_PATH, "raw", "physical", "percent_filtered.csv.gz"),
  compression="gzip", nrows=ROW_NUM)
df_lng = pd.read_csv(
  os.path.join(BASE_PATH, "raw", "physical", "percent_filtered_renamed.csv.gz"),
  compression="gzip", nrows=ROW_NUM)

In [ ]:
df_lon.head()

In [ ]:
df_lng.head()

In [ ]:
# compare
assert (df_lng["lat"] == df_lon["lat"]).all()
assert (df_lng["lng"] == df_lon["lon"]).all()
assert (df_lng["category"] == df_lon["cause_name"]).all()
assert (df_lng["value"] == df_lon["pixel_abs_prevalence"]).all()

## Remove small values
for debugging

In [ ]:
df_larger = pd.read_csv(os.path.join(BASE_PATH, "raw", "physical", "phys_aio.csv"))
df_larger = df_larger[df_larger["value"] >= 0.1]
df_larger.to_csv(os.path.join(BASE_PATH, "raw", "physical", "phys_aio_larger.csv"))

# Old Code
kept as reference

In [ ]:
# Check validity of "Percent" values
sample = pd.read_csv(DATASET_PATH, nrows=10_000, compression="gzip")
s_filtered = sample[sample['metric_name'] == "Percent"]
s_filtered['pixel_abs_prevalence'].max()
#s_filtered.head()

In [ ]:
# Analyse causes (treating them categorically since we don't expect many different causes)
# this list here seems to be a list of all possible causes (== target conditions?):
#    https://github.com/inescgu/pain-world/blob/main/prevalence.py#L8
# but this includes way more than the three we found in the data
causes = ['Osteoarthritis', 'Rheumatoid arthritis', 'Headache disorders']   # found by executing the code below
causes: Set[str] = set()
chunk_size = CHUNK_SIZE * 10      # bigger chunk size since we only care about one column and make it categorically
reader = pd.read_csv(
    DATASET_PATH,
    chunksize=chunk_size,
    usecols=["cause_name"],
    dtype={
      "cause_name": "category",
    },
    #compression="gzip",
)
last_size = len(causes)
i = 0
print(f"Start streaming with chunk size = {chunk_size}")
while True:
  try:
    chunk = next(reader)
    causes.update(chunk["cause_name"].cat.categories)
    cur_size = len(causes)
    if cur_size > last_size:
      print("causes = ", causes)
      last_size = cur_size
    i += 1
  except StopIteration:
    break
  except Exception as ex:
    print("#####################################################")
    print(f"Error occurred at cunk #{i}: ", ex)
    print("#####################################################")
print(f"DONE after {i} chunks")

In [ ]:
# Confirm structure of alternating absolute and relative numbers
# result: 
#   - for 1_000_000 it looks good
#   - for 10_000_000 it looks horrendus (79% between 0 & 1)
#   - for 100_000_000 it's even worse: 86% between 0 & 1
sample = pd.read_csv(DATASET_PATH, 
                     nrows=10_000_000, 
                     compression="gzip"
                     )

#print(sample[COL_PREVALENCE].describe())
#print(sample[COL_PREVALENCE].min(), sample[COL_PREVALENCE].max())

mask = sample[COL_PREVALENCE].between(0, 1, inclusive='left')
print(mask.value_counts())

relative = sample[mask]
duplicates = relative.duplicated(subset=["lat", "lon"])
if duplicates.any():
  print("Duplicates:")
  print(relative.loc[duplicates, ["lat", "lon", COL_PREVALENCE]])

In [ ]:
# check percentage of values < 1
sample = pd.read_csv(DATASET_PATH, nrows=10_000_000, compression="gzip")
mask = sample["pixel_abs_prevalence"].between(0, 1, inclusive='left')
#mask = sample["pixel_abs_upper"].between(0, 1, inclusive='left')
print(mask.value_counts())

In [ ]:
# inspect the presumably absolute entries with values <= 1
start_row = 16883+1
nrows = 7
misleading_abs = pd.read_csv(DATASET_PATH, skiprows=start_row, nrows=nrows, compression="gzip")
misleading_abs.head(nrows)

In [ ]:
# Create a stripped down dataset with only the needed columns and rows
chunk_size = CHUNK_SIZE
output_columns = ["lat", "lon", "cause_name", "pixel_abs_prevalence"]
reader = pd.read_csv(
    DATASET_PATH,
    chunksize=chunk_size,
    usecols=output_columns + ["metric_name"],
    dtype={
      "cause_name": "category",
    },
    compression="gzip",
)
causes: Set[str] = set()
last_size = len(causes)
i = 0
written_rows = 0
first_write = True
print(f"Start streaming with chunk size = {chunk_size}")
with gzip.open(OUTPUT_PATH, "wt", encoding="utf-8", newline="") as handle:
    while True:
      try:
        chunk = next(reader)
        filtered_chunk = chunk.loc[chunk["metric_name"] == "Percent", output_columns]
        if not filtered_chunk.empty:
          filtered_chunk.to_csv(handle, index=False, header=first_write)
          first_write = False
          written_rows += len(filtered_chunk)
        causes.update(chunk["cause_name"].cat.categories)
        cur_size = len(causes)
        if cur_size > last_size:
          print("causes = ", causes)
          last_size = cur_size
        i += 1
      except StopIteration:
        break
      except Exception as ex:
        print("#####################################################")
        print(f"Error occurred at chunk #{i}: ", ex)
        print("#####################################################")
print(f"Wrote {written_rows} rows to {OUTPUT_PATH}")
print(f"DONE after {i} chunks")

## Trying to decode the original prevalence.csv
appearently it's multiple gzipped csvs concatenated in some unknown way

In [ ]:
import gzip

In [ ]:
# sanity check if the file is valid gzip
with gzip.open(DATASET_PATH, "rb") as fin:
  try:
    while fin.read(1024 * 1024): # read 1 MB at a time:
      pass
    print("File is a valid gzip archive.")
  except Exception as e:
    print(e)

In [ ]:
# extract the first sub-file
with open(DS_CONVERTED_PATH1, "wb") as fout:
  with gzip.open(DATASET_PATH, "rb") as fin:
    try:
      while True:
        cur_chunk = fin.read(1024 * 1024) # read 1 MB at a time
        if cur_chunk:
          fout.write(cur_chunk)
        else:
          break
      print("File is a valid gzip archive.")
    except Exception as e:
      print(e)

In [ ]:
# find the start of the second sub-file
lsb = 0   # last successful byte
byte_offset = 86_948_901_888 #86_948_893_696 #86_947_921_920
chunk_size = 1 #* 1024
with gzip.open(DATASET_PATH, "rb") as fin:
  print("starting...")
  print(f"start pos = {fin.seek(byte_offset)}")
  try:
    while fin.read(chunk_size):
      lsb = fin.tell()
    print("File is a valid gzip archive.")
  except Exception as e:
    print(e)
print(f"lsb = {lsb}")

In [ ]:
# extract the second sub-file
byte_offset = 86_948_901_888
chunk_size = 1024 * 1024
with open(DS_CONVERTED_PATH2, "wb") as fout:
  with open(DATASET_PATH, "rb") as fin:
    print("starting...")
    print(f"start pos = {fin.seek(byte_offset)}")
    try:
      while True:
        cur_chunk = fin.read(chunk_size) # read 1 MB at a time
        if cur_chunk:
          fout.write(cur_chunk)
        else:
          break
      print("File is a valid gzip archive.")
    except Exception as e:
      print(e)

In [ ]:
import gzip

with gzip.open(DATASET_PATH, "rb") as f:
    i = 0
    try:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            i += 1
    except Exception as e:
        print("FAILED at MB:", i)
        print(e)

## Check filtered data

In [ ]:
# Imports
from typing import Dict, List, Optional, Set, Tuple
import gzip
import os
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Path constants
BASE_PATH = os.path.join('..', 'data')
DATASET_PATH = os.path.join(BASE_PATH, 'raw', 'physical', 'percent_filtered.csv.gz')

In [ ]:
sample = pd.read_csv(DATASET_PATH, nrows=10_000, compression="gzip")
print(sample.dtypes)
sample.head()

In [ ]:
plt.boxplot(sample['pixel_abs_prevalence'])
print(f"max = {sample['pixel_abs_prevalence'].max()}")